In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import time

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

SEED = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
RESULT_DIR = PROJECT_ROOT / "results" / "metrics"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

X_train_text = train_df["clean_url"].fillna("").astype(str)
X_val_text = val_df["clean_url"].fillna("").astype(str)
X_test_text = test_df["clean_url"].fillna("").astype(str)

y_train = train_df["label_binary"]
y_val = val_df["label_binary"]
y_test = test_df["label_binary"]

In [3]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    max_features=30000,
    min_df=3,
    dtype=np.float32,
    lowercase=False
)

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

In [4]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=25,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=SEED,
    n_jobs=-1
)

start_time = time.perf_counter()

rf_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

In [5]:
start_time = time.perf_counter()

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

inference_time = time.perf_counter() - start_time

metrics = {
    "model": "Random Forest",
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "training_seconds": round(training_time, 2),
    "inference_seconds": round(inference_time, 2),
    "test_samples": len(y_test)
}

joblib.dump(rf_model, MODEL_DIR / "random_forest.joblib")
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")

with open(RESULT_DIR / "random_forest_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [6]:
print("=" * 60)
print("RANDOM FOREST BASELINE RESULTS")
print("=" * 60)

print(f"Training samples : {len(y_train):,}")
print(f"Testing samples  : {len(y_test):,}")
print(f"TF-IDF features  : {X_train.shape[1]:,}")

print("-" * 60)

print(f"Accuracy         : {metrics['accuracy']:.4f}")
print(f"Precision        : {metrics['precision']:.4f}")
print(f"Recall           : {metrics['recall']:.4f}")
print(f"F1-score         : {metrics['f1_score']:.4f}")
print(f"ROC-AUC          : {metrics['roc_auc']:.4f}")

print("-" * 60)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("-" * 60)
print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "Malicious"],
    zero_division=0
))

print(f"Training time    : {training_time:.2f} seconds")
print(f"Inference time   : {inference_time:.2f} seconds")
print("Model saved      : models/random_forest.joblib")
print("=" * 60)

RANDOM FOREST BASELINE RESULTS
Training samples : 448,779
Testing samples  : 96,167
TF-IDF features  : 30,000
------------------------------------------------------------
Accuracy         : 0.9347
Precision        : 0.9858
Recall           : 0.8154
F1-score         : 0.8925
ROC-AUC          : 0.9884
------------------------------------------------------------
Confusion Matrix:
[[63836   375]
 [ 5900 26056]]
------------------------------------------------------------
              precision    recall  f1-score   support

      Benign       0.92      0.99      0.95     64211
   Malicious       0.99      0.82      0.89     31956

    accuracy                           0.93     96167
   macro avg       0.95      0.90      0.92     96167
weighted avg       0.94      0.93      0.93     96167

Training time    : 89.90 seconds
Inference time   : 1.39 seconds
Model saved      : models/random_forest.joblib
